# 004-reposed-frames — correlator walkthrough

# Imports

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from uavloc.data.drone_frames import load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import build_features, correlator_config
from uavloc.features.segmentation import CLASSES
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.correlator import PositionCorrelator, class_ncc_maps, crop_border, drone_class_shares
from uavloc.plotting.correlation import (CORRECTED_COLOURS, PRIOR_COLOUR, SEG_PALETTE, TRUE_COLOUR, class_colours,
                                         class_overlay, footprint_rectangle, mark_points, plot_class_surface,
                                         plot_heatmap, plot_match, plot_overview, show_patch)
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "004-reposed-frames"
RUN = "004.2"
VGG_EXP, VGG_RUN = "004-reposed-frames", "004.1"
NOISE_M = 10.0
DEVICE = "mps"

CASES = None                      # None -> automatic picks below; or [[frame, bearing_deg], ...] from the records
BLOCK_GAP_MS = 5000               # frames further apart than this belong to different passes (blocks)
LOSING_BLOCKS = [1, 2]            # where 002.2 loses to VGG11 (report): worst reachable draw of each
OVERVIEW_MARGIN_M = 85.0
PAIRS_DIR = "../archive/uav_localization_2025/data/drone/images_2_v3_renamed_small_easy"   # 2025 training pairs: NNNN_<ms>_drone/sat_patch.jpg

SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
config = next(r for r in load_runs(EXP) if r["run"] == RUN)["config"]
vgg_config = next(r for r in load_runs(VGG_EXP) if r["run"] == VGG_RUN)["config"]
frames = {f.name: f for f in load_frames(config["frames_dataset"])}
satellite = GeoImage.load(REPO_ROOT / config["sat_image"], LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
class_weights = tuple(config["class_weights"])

In [ ]:
def load_records(exp: str, run_id: str) -> pd.DataFrame:
    records = pd.read_csv(experiment_dir(exp) / "artifacts" / f"records-{run_id}.csv")
    return records[records["noise_m"] == NOISE_M].set_index(["frame", "bearing_deg"])

In [ ]:
def time_blocks(names: list[str]) -> pd.Series:
    """Pass index per frame: a new block starts after a gap longer than BLOCK_GAP_MS."""
    stamps = pd.Series({n: int(n.split("-")[1]) for n in names}).sort_values()
    return (stamps.diff() > BLOCK_GAP_MS).cumsum()

In [ ]:
draws = load_records(EXP, RUN).join(load_records(VGG_EXP, VGG_RUN)[["error_m"]], rsuffix="_vgg11").reset_index()
draws["block"] = draws["frame"].map(time_blocks(list(frames)))

In [ ]:
def pick_cases() -> pd.DataFrame:
    """Worst reachable draw in each losing block, then a typical and the best reachable draw elsewhere."""
    if CASES is not None:
        return pd.concat([draws[(draws["frame"] == f) & np.isclose(draws["bearing_deg"], b)] for f, b in CASES])
    reachable = draws[draws["reachable"]]
    losing = [reachable[reachable["block"] == b].nlargest(1, "error_m") for b in LOSING_BLOCKS]
    others = reachable[~reachable["block"].isin(LOSING_BLOCKS)].sort_values("error_m")
    typical = others.iloc[[len(others) // 2]]
    return pd.concat([*losing, typical, others.head(1)]).assign(why=["worst, block 1", "worst, block 2", "typical", "best"][:len(losing) + 2])

In [ ]:
cases = pick_cases().reset_index(drop=True)
cases[["why", "frame", "block", "bearing_deg", "error_m", "error_m_vgg11", "peak_z"]]

# Model

In [ ]:
captured = {}
correlator = PositionCorrelator(build_features(config, satellite, DEVICE), correlator_config(config),
                                features_hook=lambda d, s: captured.update(drone=d, satellite=s))
vgg11 = PositionCorrelator(build_features(vgg_config, satellite, DEVICE), correlator_config(vgg_config))

# Run

In [ ]:
def trace(case: pd.Series) -> dict:
    """Replay one logged draw and keep every intermediate the steps below draw."""
    frame, prior = frames[case["frame"]], LatLon(case["prior_lat"], case["prior_lon"])
    correction = correlator.correct(frame, prior)
    cfg = correlator.config
    drone, _, _ = crop_border(captured["drone"], cfg.drone_feature_crop_pct)
    sat, _, _ = crop_border(captured["satellite"], cfg.sat_feature_crop_pct)
    footprint_m = correlator.footprint_m(frame)
    return {"case": case, "frame": frame, "prior": prior, "correction": correction, "footprint_m": footprint_m,
            "drone_map": captured["drone"], "satellite_map": captured["satellite"],
            "class_maps": class_ncc_maps(drone, sat, class_weights), "shares": drone_class_shares(captured["drone"]),
            "vgg11": vgg11.correct(frame, prior).position,
            "patch_rgb": satellite.crop_rotated(prior, frame.yaw_deg, *correction.patch_size_m),
            "true_offset_m": correlator.patch_offset_m(prior, frame.position, frame.yaw_deg),
            "map_at_truth": correlator.features.satellite_patch(frame.position, frame.yaw_deg, *footprint_m),
            "map_at_answer": correlator.features.satellite_patch(correction.position, frame.yaw_deg, *footprint_m)}

In [ ]:
traces = [trace(case) for _, case in cases.iterrows()]
pd.DataFrame([{"case": t["case"]["why"], "replayed_m": haversine_m(t["correction"].position, t["frame"].position),
               "logged_m": t["case"]["error_m"]} for t in traces])

In [ ]:
def row_label(t: dict) -> str:
    c = t["case"]
    return (f"{c['why']}\n{c['frame'][-7:]}  block {c['block']}\nalt {t['frame'].altitude_m:.0f} m  yaw {t['frame'].yaw_deg:.0f}\n"
            f"error {c['error_m']:.1f} m (VGG11 {c['error_m_vgg11']:.1f})")

In [ ]:
def share_text(shares: np.ndarray) -> str:
    return ", ".join(f"{CLASSES[c]} {s:.0%}" for c, s in enumerate(shares) if s >= 0.005)

# Step 1: where the drone really is, where it was said to be, and both answers

In [ ]:
fig, axes = plt.subplots(1, len(traces), figsize=(6 * len(traces), 6))
for ax, t in zip(axes, traces):
    plot_overview(ax, satellite, t["frame"], t["prior"], {f"{RUN} weighted NCC": t["correction"].position, "VGG11": t["vgg11"]},
                  t["correction"].patch_size_m, OVERVIEW_MARGIN_M)
    ax.set_title(row_label(t).replace("\n", " | "), fontsize=8)
plt.tight_layout(); plt.show()

# Step 2: the two inputs

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(16, 3.8 * len(traces)))
for row, t in zip(axes, traces):
    row[0].imshow(t["frame"].image()); row[0].set_xticks([]); row[0].set_yticks([])
    row[0].set_ylabel(row_label(t), fontsize=8); row[0].set_title("drone frame (top of image = direction of flight)", fontsize=9)
    show_patch(row[1], t["patch_rgb"], t["correction"].patch_size_m, "satellite patch cut around the prior, turned to the heading")
    mark_points(row[1], t["correction"], t["true_offset_m"], CORRECTED_COLOURS[0])
    row[1].add_patch(footprint_rectangle(t["true_offset_m"], t["footprint_m"], TRUE_COLOUR))
axes[0, 1].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

# Step 2b: drone frame vs satellite at the logged pose, and the 2025 dataset pair

In [ ]:
fig, axes = plt.subplots(len(traces), 5, figsize=(27, 3.2 * len(traces)))
for row, t in zip(axes, traces):
    stamp = t["frame"].name.split("-")[1]
    pair = sorted((REPO_ROOT / PAIRS_DIR).glob(f"*_{stamp}_sat_patch.jpg"), key=lambda p: int(p.name.split("_")[0]))
    panels = [(np.asarray(t["frame"].image()), f"drone frame {stamp}"),
              (np.asarray(Image.open(str(pair[0]).replace("_sat_patch", "_drone"))) if pair else None, f"2025 pair: drone ({pair[0].name.split('_')[0] if pair else 'none'})"),
              (np.asarray(Image.open(pair[0])) if pair else None, f"2025 pair: satellite patch ({len(pair)} found)"),
              (satellite.crop_rotated(t["frame"].position, t["frame"].yaw_deg, *t["footprint_m"]), "orthophoto at logged position + yaw"),
              (class_colours(t["map_at_truth"], list(range(len(CLASSES)))), "your map at logged position + yaw")]
    for ax, (image, title) in zip(row, panels):
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=9)
        ax.imshow(image) if image is not None else ax.text(.5, .5, "no 2025 pair", ha="center", va="center")
    row[0].set_ylabel(row_label(t), fontsize=8)
plt.tight_layout(); plt.show()

# Step 3: drone side, what the model sees in the frame

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(13, 3.4 * len(traces)))
for row, t in zip(axes, traces):
    row[0].imshow(t["frame"].image()); row[0].set_ylabel(row_label(t), fontsize=8)
    row[1].imshow(class_colours(t["drone_map"], list(range(len(CLASSES)))))
    row[1].set_title(f"drone classes: {share_text(t['shares'])}", fontsize=9)
    for ax in row: ax.set_xticks([]); ax.set_yticks([])
handles = [plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))]
fig.legend(handles, CLASSES, loc="lower center", ncol=len(CLASSES)); plt.tight_layout(rect=(0, .03, 1, 1)); plt.show()

# Step 4: satellite side, your map inside the patch

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(16, 3.8 * len(traces)))
for row, t in zip(axes, traces):
    size_m = t["correction"].patch_size_m
    show_patch(row[0], t["patch_rgb"], size_m, "satellite patch"); row[0].set_ylabel(row_label(t), fontsize=8)
    show_patch(row[1], class_colours(t["satellite_map"], list(range(len(CLASSES)))), size_m,
               "hand-labelled map, same patch (green box = where the drone really is)")
    for ax in row:
        ax.add_patch(footprint_rectangle(t["true_offset_m"], t["footprint_m"], TRUE_COLOUR))
        ax.add_patch(footprint_rectangle((0, 0), t["footprint_m"], PRIOR_COLOUR, "--"))
plt.tight_layout(); plt.show()

# Step 5: one correlation surface per class

In [ ]:
weighted = [c for c, w in enumerate(class_weights) if w > 0]
fig, axes = plt.subplots(len(traces), len(weighted), figsize=(5.2 * len(weighted), 3.4 * len(traces)))
for row, t in zip(axes, traces):
    for ax, c in zip(row, weighted):
        title = f"{CLASSES[c]} (weight {class_weights[c]:g})"
        if c in t["class_maps"]:
            surface = t["class_maps"][c]
            shown = plot_class_surface(ax, surface, t["correction"], t["true_offset_m"], CORRECTED_COLOURS[0],
                                       f"{title}\nNCC {surface.min():.2f} .. {surface.max():.2f}")
            fig.colorbar(shown, ax=ax, shrink=.8)
        else:
            ax.text(.5, .5, f"{title}\nskipped: {t['shares'][c]:.1%} of drone pixels\n(needs 1%)", ha="center", va="center")
            ax.set_xticks([]); ax.set_yticks([])
    row[0].set_ylabel(row_label(t), fontsize=8)
plt.tight_layout(); plt.show()

# Step 6: weighted sum of the class surfaces, and the answer

In [ ]:
fig, axes = plt.subplots(len(traces), 2, figsize=(16, 3.8 * len(traces)))
for row, t in zip(axes, traces):
    c = t["correction"]
    shown = plot_heatmap(row[0], t["patch_rgb"], c, t["true_offset_m"], CORRECTED_COLOURS[0], f"combined surface (z-score), peak z {c.peak_z:.2f}")
    fig.colorbar(shown, ax=row[0], shrink=.8)
    plot_match(row[1], t["patch_rgb"], c, t["true_offset_m"], t["footprint_m"], CORRECTED_COLOURS[0],
               f"red box = chosen footprint, green = true: {haversine_m(c.position, t['frame'].position):.1f} m apart")
    row[0].set_ylabel(row_label(t), fontsize=8)
axes[0, 1].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

# Step 7: why the chosen place beat the true one

In [ ]:
def class_scores(t: dict, offset_m: tuple[float, float]) -> dict[str, float]:
    """Each taking-part class's NCC at the heatmap cell nearest `offset_m`; NaN outside the search range."""
    cell = t["correction"].heatmap_cell(offset_m)
    return {CLASSES[c]: (float(m[cell]) if cell else np.nan) for c, m in t["class_maps"].items()}

In [ ]:
fig, axes = plt.subplots(len(traces), 5, figsize=(29, 3.2 * len(traces)), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.1]})
track, woody = CLASSES.index("track"), CLASSES.index("woody")
for row, t in zip(axes, traces):
    at_truth, at_answer = class_scores(t, t["true_offset_m"]), class_scores(t, t["correction"].offset_m)
    for ax, fmap, title in zip(row[:3], [t["drone_map"], t["map_at_truth"], t["map_at_answer"]],
                               ["drone classes", "your map under the TRUE footprint", "your map under the CHOSEN footprint"]):
        ax.imshow(class_colours(fmap, list(range(len(CLASSES))))); ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    row[3].imshow(class_overlay(t["drone_map"], t["map_at_truth"], track, woody)); row[3].set_xticks([]); row[3].set_yticks([])
    row[3].set_title("drone over map at TRUE place\ntrack: yellow drone, cyan map, white both", fontsize=9)
    names, x = list(at_truth), np.arange(len(at_truth))
    row[4].bar(x - .2, [at_truth[n] for n in names], .4, color=TRUE_COLOUR, label="at true place")
    row[4].bar(x + .2, [at_answer[n] for n in names], .4, color=CORRECTED_COLOURS[0], label="at chosen place")
    row[4].set_xticks(x, [f"{n}\n(w {class_weights[CLASSES.index(n)]:g})" for n in names]); row[4].axhline(0, color="k", lw=.5)
    row[4].set_ylim(-1, 1); row[4].set_ylabel("class NCC"); row[4].legend(fontsize=7)
    row[0].set_ylabel(row_label(t), fontsize=8)
plt.tight_layout(); plt.show()